# LLMOps: API Cost, Latency, and Prompt Version Management
**Course:** MLOps  |  **Experiential Learning Assignment-2**  |  **Scenario:** EduAssist AI (academic question-answering assistant)

This notebook covers all three tasks of the assignment:

| Task | Marks | Section |
|---|---|---|
| 1. Call an LLM and measure latency | 2 | Task 1 |
| 2. Count tokens and calculate cost | 3 | Task 2 |
| 3. Prompt versioning and MLflow evaluation | 5 | Task 3 |

**Model used.** API access is not used; as the assignment allows, a locally hosted open-source model (`Qwen/Qwen2.5-1.5B-Instruct`, run with Hugging Face `transformers` on CPU) is used instead. Token counts come from the model's own tokenizer, and cost is estimated from a published hosting price (see the config cell). No API key is required; an optional `HF_TOKEN` environment variable is read only if set (it is never hard-coded).

## 0. Setup

In [ ]:
# Install the required libraries only if something is missing (see requirements.txt).
import importlib.util, subprocess, sys
_needed = ["mlflow", "transformers", "torch", "pandas"]
if any(importlib.util.find_spec(m) is None for m in _needed):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
    print("Dependencies installed.")
else:
    print("All required libraries are already installed.")

In [ ]:
import os, json, time, platform, difflib
from datetime import datetime
from pathlib import Path

import pandas as pd
import torch, transformers, mlflow
from transformers import AutoTokenizer, AutoModelForCausalLM
from mlflow.entities import AssessmentSource, AssessmentSourceType

# ---------------- Configuration (everything that may change is here) ----------------
MODEL_ID = os.environ.get("EDUASSIST_MODEL", "Qwen/Qwen2.5-1.5B-Instruct")
HF_TOKEN = os.environ.get("HF_TOKEN")      # optional; read from the environment, never hard-coded
MAX_NEW_TOKENS = 256                       # same cap for every request
DECODING = "greedy (do_sample=False, i.e. temperature 0)"

# Published hosting price used to ESTIMATE cost of the locally hosted model (no real bill exists).
PRICE_INPUT_PER_M = 0.10                   # USD per 1,000,000 input tokens
PRICE_OUTPUT_PER_M = 0.10                  # USD per 1,000,000 output tokens
CURRENCY = "USD"
PRICING_DATE = "retrieved 2026-10-31 (source page last updated 2026-07-21)"
PRICING_SOURCE = ("Hosted-API list price for Qwen2.5-1.5B-Instruct: "
                  "https://pricepertoken.com/pricing-page/model/qwen-qwen2.5-1.5b-instruct "
                  "(cross-checked with the Fireworks listing on getmaxim.ai: $0.10 in / $0.10 out per 1M tokens)")
REQUESTS_PER_MONTH = 10_000

EXPERIMENT_NAME = "eduassist_llmops_assignment"
PROMPT_NAME = "eduassist_qa_prompt"        # one registry entry holding versions 1, 2, 3
TRACKING_URI = f"sqlite:///{Path('mlflow.db').resolve()}"

torch.set_num_threads(os.cpu_count() or 1)
if MODEL_ID != "Qwen/Qwen2.5-1.5B-Instruct":
    print("WARNING: model changed - update the price settings above to match the new model.")

print("Python        :", platform.python_version())
print("torch         :", torch.__version__, "| transformers:", transformers.__version__, "| mlflow:", mlflow.__version__)
print("pandas        :", pd.__version__, "| CPU threads:", torch.get_num_threads(), "| CUDA:", torch.cuda.is_available())
print("Model         :", MODEL_ID, "| decoding:", DECODING, "| max_new_tokens:", MAX_NEW_TOKENS)
print(f"Pricing       : {PRICE_INPUT_PER_M} {CURRENCY}/M input, {PRICE_OUTPUT_PER_M} {CURRENCY}/M output, {PRICING_DATE}")
print("Pricing source:", PRICING_SOURCE)

### Questions and prompt versions
Five academic questions (the same five are used in every task) and the three prompt versions exactly as given in the assignment.

In [ ]:
QUESTIONS = [
    ("Q1", "What is overfitting in machine learning?"),
    ("Q2", "Explain the difference between supervised and unsupervised learning."),
    ("Q3", "What is gradient descent and why is it used?"),
    ("Q4", "What is the bias-variance tradeoff?"),
    ("Q5", "What is the difference between precision and recall?"),
]

V1_TEMPLATE = """Answer the following student question clearly.
Question: {{question}}"""

V2_TEMPLATE = """You are an academic assistant.
Answer the following student question in a clear, concise, and accurate manner.
Use simple language suitable for undergraduate students.
Question: {{question}}"""

V3_TEMPLATE = """You are an academic assistant.
Answer the student question and return ONLY valid JSON.
Do not include markdown, explanations outside JSON, or code fences.
Use this exact JSON schema:
{
  "answer": "string",
  "confidence": "high | medium | low",
  "key_points": ["string"]
}
Question: {{question}}"""

print(len(QUESTIONS), "questions loaded")

### Load the model and define the LLM call helper
The helper records start/end time, latency, response status and token usage for every request. Model loading and one short warm-up generation happen **before** any measurement, so one-off start-up cost does not distort latency.

In [ ]:
t0 = time.perf_counter()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
try:
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, token=HF_TOKEN, dtype=torch.float32)
except TypeError:                                    # older transformers versions
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, token=HF_TOKEN, torch_dtype=torch.float32)
model.eval()
print(f"Model loaded in {time.perf_counter() - t0:.1f}s | revision: {getattr(model.config, '_commit_hash', 'n/a')}")


def call_llm(prompt_text):
    """Send one prompt to the local model; return text, token usage, timing and status."""
    start = datetime.now()
    t_start = time.perf_counter()
    text, n_in, n_out, status = "", 0, 0, "success"
    try:
        chat = [{"role": "user", "content": prompt_text}]
        rendered = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(rendered, return_tensors="pt", add_special_tokens=False)
        n_in = int(inputs["input_ids"].shape[1])           # input tokens (includes chat-template tokens)
        with torch.inference_mode():
            out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                                 temperature=None, top_p=None, top_k=None,
                                 pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id)
        new_ids = out[0][n_in:]
        n_out = int(new_ids.shape[0])                      # output tokens
        text = tokenizer.decode(new_ids, skip_special_tokens=True)
    except Exception as e:                                 # a failed request is recorded, not fatal
        status = f"error: {type(e).__name__}: {e}"
    t_end = time.perf_counter()
    return {"response_text": text, "input_tokens": n_in, "output_tokens": n_out,
            "total_tokens": n_in + n_out, "status": status,
            "start_time": start.isoformat(timespec="milliseconds"),
            "end_time": datetime.now().isoformat(timespec="milliseconds"),
            "latency_seconds": t_end - t_start}


# warm-up (not recorded)
_ = call_llm("Say hello.")
print("Warm-up done.")

## Task 1: Call an LLM and Measure Latency (2 marks)
Five academic questions are sent to the model with **Prompt v1 (basic)**. For each request the question ID, prompt version, model name, start time, end time, latency and response status are recorded.

In [ ]:
rows = []
for qid, question in QUESTIONS:
    r = call_llm(V1_TEMPLATE.replace("{{question}}", question))
    rows.append({"question_id": qid, "prompt_version": "v1", "model": MODEL_ID, **r})
    print(f"{qid}: {r['latency_seconds']:.2f}s | in={r['input_tokens']} out={r['output_tokens']} | {r['status']}")

task1 = pd.DataFrame(rows)
task1["latency_seconds"] = task1["latency_seconds"].round(3)
print()
display(task1[["question_id", "prompt_version", "model", "start_time", "end_time", "latency_seconds", "status"]])

In [ ]:
ok = task1[task1["status"] == "success"]["latency_seconds"]
if len(ok) < len(task1):
    print(f"NOTE: {len(task1) - len(ok)} request(s) failed and are excluded from the statistics.")

latency_summary = pd.Series({
    "mean": ok.mean(), "median": ok.median(), "min": ok.min(),
    "max": ok.max(), "std (sample)": ok.std(ddof=1),
}, name="latency_seconds").round(3)
display(latency_summary.to_frame())

In [ ]:
slow = task1.loc[task1["latency_seconds"].idxmax()]
r_corr = task1["latency_seconds"].corr(task1["output_tokens"])
hit_cap = int(slow["output_tokens"]) >= MAX_NEW_TOKENS

corr_text = ""
if pd.notna(r_corr):
    if r_corr >= 0.7:
        corr_text = (f"Across the five requests, latency and output length are strongly correlated (r = {r_corr:.2f}), which fits how an "
                     f"autoregressive model works: it produces one token at a time, so longer answers take longer, and on a CPU-only "
                     f"machine this per-token cost is high. ")
    else:
        corr_text = (f"Across the five requests, latency and output length are only weakly correlated (r = {r_corr:.2f}), so output length "
                     f"alone does not explain the differences. ")

interpretation = (
    f"The slowest request was {slow['question_id']} at {slow['latency_seconds']:.2f} s, against a mean of "
    f"{ok.mean():.2f} s and a fastest request of {ok.min():.2f} s. It generated {int(slow['output_tokens'])} output tokens"
    f"{' (it reached the max_new_tokens cap of ' + str(MAX_NEW_TOKENS) + ')' if hit_cap else ''}, "
    f"while the average request generated {task1['output_tokens'].mean():.0f}. "
    + corr_text +
    f"Other possible contributors are a longer prompt (more input tokens to process) and run-to-run variation from other "
    f"processes sharing the same CPU."
)
print(interpretation)

## Task 2: Count Tokens and Calculate Cost (3 marks)
Token usage is taken from each response (input tokens from the tokenizer, output tokens from the generated ids). Cost uses the assignment formulas with the published hosting price from the config cell:

- Input Cost = Input Tokens / 1,000,000 × Input Price per Million Tokens
- Output Cost = Output Tokens / 1,000,000 × Output Price per Million Tokens
- Total Cost per Request = Input Cost + Output Cost
- Projected Monthly Cost = Average Cost per Request × 10,000

*A locally hosted model produces no real bill, so the cost is an estimate based on the published hosting price of the same model (see the config cell for the date, currency and source).*

In [ ]:
task2 = task1.copy()
task2["input_cost"]  = task2["input_tokens"]  / 1_000_000 * PRICE_INPUT_PER_M
task2["output_cost"] = task2["output_tokens"] / 1_000_000 * PRICE_OUTPUT_PER_M
task2["total_cost"]  = task2["input_cost"] + task2["output_cost"]

cost_table = task2[["question_id", "input_tokens", "output_tokens", "total_tokens",
                    "input_cost", "output_cost", "total_cost"]].copy()
cost_table = cost_table.rename(columns={"input_cost": f"input_cost_{CURRENCY}",
                                        "output_cost": f"output_cost_{CURRENCY}",
                                        "total_cost": f"total_cost_{CURRENCY}"})
display(cost_table.style.format({f"input_cost_{CURRENCY}": "{:.8f}", f"output_cost_{CURRENCY}": "{:.8f}",
                                 f"total_cost_{CURRENCY}": "{:.8f}"}))

avg_cost = task2["total_cost"].mean()
monthly = avg_cost * REQUESTS_PER_MONTH
print(f"Average cost per request       : {avg_cost:.8f} {CURRENCY}")
print(f"Projected monthly cost ({REQUESTS_PER_MONTH:,} requests): {monthly:.4f} {CURRENCY}")

In [ ]:
# Cost-optimisation recommendation: shorten responses (output tokens are generated one by one and, here, billed at the same rate as input).
out_share = task2["output_cost"].sum() / task2["total_cost"].sum()
CAP = 128
capped_out = task2["output_tokens"].clip(upper=CAP)
capped_cost = (task2["input_tokens"] / 1e6 * PRICE_INPUT_PER_M + capped_out / 1e6 * PRICE_OUTPUT_PER_M).mean()
saving = 1 - capped_cost / avg_cost

recommendation = (
    f"Recommendation - shorten responses. Output tokens make up {out_share:.0%} of the cost of these requests "
    f"(average {task2['output_tokens'].mean():.0f} output vs {task2['input_tokens'].mean():.0f} input tokens). "
    f"Asking for concise answers (Prompt v2 does this) and capping output length would reduce both cost and latency; "
    f"as an illustration, capping each answer at {CAP} tokens would lower the average cost per request from "
    f"{avg_cost:.8f} to {capped_cost:.8f} {CURRENCY} (about {saving:.0%}), i.e. the monthly projection from "
    f"{monthly:.4f} to {capped_cost * REQUESTS_PER_MONTH:.4f} {CURRENCY}. The trade-off is that a hard cap can cut answers "
    f"short, so it should be checked against answer quality."
)
print(recommendation)

In [ ]:
# Save the Task 1 + Task 2 results.
csv1 = task2[["question_id", "prompt_version", "model", "start_time", "end_time", "latency_seconds", "status",
              "input_tokens", "output_tokens", "total_tokens", "input_cost", "output_cost", "total_cost"]].copy()
csv1 = csv1.rename(columns={"input_cost": f"input_cost_{CURRENCY}", "output_cost": f"output_cost_{CURRENCY}",
                            "total_cost": f"total_cost_{CURRENCY}"})
csv1.to_csv("latency_tokens_cost.csv", index=False)
print("Saved latency_tokens_cost.csv", csv1.shape)

## Task 3: Prompt Versioning and MLflow Evaluation (5 marks)
1. Create the MLflow experiment `eduassist_llmops_assignment`.
2. Register Prompt v1, v2 and v3 in the MLflow Prompt Registry (one prompt, three versions, each with a commit message).
3. Run every version on the same five questions and log `avg_latency_seconds`, `avg_input_tokens`, `avg_output_tokens`, `avg_cost_per_request` and `valid_json`.
4. Compare the versions and promote the best one with the `production` alias.

In [ ]:
mlflow.set_tracking_uri(TRACKING_URI)
experiment = mlflow.set_experiment(EXPERIMENT_NAME)
print("Tracking URI :", TRACKING_URI)
print("Experiment   :", experiment.name, "| id:", experiment.experiment_id)

### 3.1 Register the three prompt versions

In [ ]:
TEMPLATES = {1: V1_TEMPLATE, 2: V2_TEMPLATE, 3: V3_TEMPLATE}
COMMIT_MESSAGES = {
    1: "v1 basic: one-line instruction, no role, no length or format constraints (baseline)",
    2: "v2 structured: adds academic-assistant role, concise and accurate tone, simple language for undergraduates",
    3: "v3 json output: strict JSON-only reply with fixed schema (answer, confidence, key_points); no markdown or code fences",
}

def _already_registered(version):
    try:
        return mlflow.genai.load_prompt(f"prompts:/{PROMPT_NAME}/{version}")
    except Exception:
        return None

for v in (1, 2, 3):
    existing = _already_registered(v)
    if existing is not None:                       # makes the notebook safe to re-run
        print(f"v{v} already registered (version {existing.version}) - reusing it")
        continue
    pv = mlflow.genai.register_prompt(name=PROMPT_NAME, template=TEMPLATES[v], commit_message=COMMIT_MESSAGES[v])
    print(f"Registered {PROMPT_NAME} version {pv.version}: {COMMIT_MESSAGES[v]}")

### 3.2 Run each prompt version on the same five questions
Every request is traced, and a boolean `valid_json` feedback (does `json.loads` accept the response exactly as returned?) is attached to each trace. MLflow averages boolean feedback, which is the JSON-validity pass rate:

valid_json = Valid JSON Responses / Total Responses

A response wrapped in a markdown code fence or followed by extra text is counted as **not valid**. `schema_ok` (extra, for the quality discussion) additionally checks the exact schema: keys `answer` (string), `confidence` (high/medium/low) and `key_points` (list of strings).

In [ ]:
def check_json(text):
    """Return (valid_json, schema_ok) for one raw model response."""
    try:
        obj = json.loads(text.strip())
    except Exception:
        return False, False
    schema_ok = (isinstance(obj, dict)
                 and set(obj.keys()) == {"answer", "confidence", "key_points"}
                 and isinstance(obj["answer"], str)
                 and obj["confidence"] in {"high", "medium", "low"}
                 and isinstance(obj["key_points"], list)
                 and all(isinstance(k, str) for k in obj["key_points"]))
    return True, schema_ok


@mlflow.trace(name="eduassist_qa")
def answer_question(prompt_version, question_id, prompt_text):
    mlflow.update_current_trace(tags={"prompt_version": prompt_version, "question_id": question_id})
    return call_llm(prompt_text)

In [ ]:
eval_rows, version_metrics = [], {}

for v in (1, 2, 3):
    label = f"v{v}"
    with mlflow.start_run(run_name=f"prompt_{label}"):
        prompt = mlflow.genai.load_prompt(f"prompts:/{PROMPT_NAME}/{v}")      # loaded inside the run, so MLflow links it
        mlflow.log_params({"prompt_name": PROMPT_NAME, "prompt_version": label, "model": MODEL_ID,
                           "max_new_tokens": MAX_NEW_TOKENS, "decoding": DECODING,
                           "price_input_per_M": PRICE_INPUT_PER_M, "price_output_per_M": PRICE_OUTPUT_PER_M,
                           "currency": CURRENCY})
        trace_ids, run_rows = [], []
        for qid, question in QUESTIONS:
            res = answer_question(label, qid, prompt.format(question=question))
            trace_ids.append(mlflow.get_last_active_trace_id())
            valid, schema_ok = check_json(res["response_text"])
            cost = (res["input_tokens"] / 1e6 * PRICE_INPUT_PER_M) + (res["output_tokens"] / 1e6 * PRICE_OUTPUT_PER_M)
            run_rows.append({"question_id": qid, "prompt_version": label, "model": MODEL_ID,
                             "latency_seconds": round(res["latency_seconds"], 3), "status": res["status"],
                             "input_tokens": res["input_tokens"], "output_tokens": res["output_tokens"],
                             "total_tokens": res["total_tokens"], f"total_cost_{CURRENCY}": cost,
                             "valid_json": valid, "schema_ok": schema_ok, "response_text": res["response_text"]})
            print(f"{label} {qid}: {res['latency_seconds']:.1f}s | out={res['output_tokens']} | valid_json={valid}")

        # per-response boolean feedback -> MLflow aggregates it into the pass rate
        mlflow.flush_trace_async_logging()
        for tid, row in zip(trace_ids, run_rows):
            mlflow.log_feedback(trace_id=tid, name="valid_json", value=bool(row["valid_json"]),
                                source=AssessmentSource(source_type=AssessmentSourceType.CODE, source_id="json.loads"))

        d = pd.DataFrame(run_rows)
        good = d[d["status"] == "success"]
        version_metrics[label] = {
            "avg_latency_seconds": good["latency_seconds"].mean(),
            "avg_input_tokens": good["input_tokens"].mean(),
            "avg_output_tokens": good["output_tokens"].mean(),
            "avg_cost_per_request": good[f"total_cost_{CURRENCY}"].mean(),
            "valid_json": d["valid_json"].mean(),            # all responses count; failed requests are invalid
            "schema_ok": d["schema_ok"].mean(),
        }
        for name in ("avg_latency_seconds", "avg_input_tokens", "avg_output_tokens", "avg_cost_per_request", "valid_json"):
            mlflow.log_metric(name, float(version_metrics[label][name]))
        eval_rows.extend(run_rows)
    print(f"-> {label} metrics logged: " + ", ".join(f"{k}={val:.4g}" for k, val in version_metrics[label].items()), "\n")

### 3.3 Compare the versions and choose the best one

In [ ]:
metrics = pd.DataFrame(version_metrics).T
# Selection rule: highest valid_json, then highest schema_ok, then lowest average latency.
order = metrics.sort_values(["valid_json", "schema_ok", "avg_latency_seconds"], ascending=[False, False, True])
best = order.index[0]
if metrics.loc[best, "valid_json"] == 0:
    print("WARNING: no prompt version produced valid JSON - review the responses before promoting anything.")

comparison = pd.DataFrame({
    "Prompt Version": metrics.index,
    "Avg Latency": metrics["avg_latency_seconds"].map("{:.2f} s".format).values,
    "Avg Input Tokens": metrics["avg_input_tokens"].map("{:.1f}".format).values,
    "Avg Output Tokens": metrics["avg_output_tokens"].map("{:.1f}".format).values,
    "Avg Cost": metrics["avg_cost_per_request"].map(lambda x: f"{x:.8f} {CURRENCY}").values,
    "Valid JSON Rate": metrics["valid_json"].map("{:.0%}".format).values,
    "Recommended?": ["Yes" if v == best else "No" for v in metrics.index],
})
display(comparison)
print("Schema-compliance rate (extra check):", {k: f"{v:.0%}" for k, v in metrics["schema_ok"].items()})
print("Best version by the selection rule  :", best)

In [ ]:
# Prompt diff evidence (the MLflow UI shows the same diff: Prompts > eduassist_qa_prompt > Compare).
def show_diff(a, b):
    print(f"--- {a} -> {b} ---")
    for line in difflib.unified_diff(TEMPLATES[int(a[1])].splitlines(), TEMPLATES[int(b[1])].splitlines(),
                                     fromfile=a, tofile=b, lineterm=""):
        print(line)
    print()
show_diff("v1", "v2")
show_diff("v2", "v3")

### 3.4 Promote the best version with the `production` alias

In [ ]:
best_version = int(best[1])
mlflow.genai.set_prompt_alias(PROMPT_NAME, alias="production", version=best_version)

prod = mlflow.genai.load_prompt(f"prompts:/{PROMPT_NAME}@production")      # verify by loading through the alias
print(f"Alias 'production' now points to {PROMPT_NAME} version {prod.version}")
print("Commit message:", prod.commit_message)
print("--- template ---")
print(prod.template)
assert prod.version == best_version

In [ ]:
# Save per-response evaluation results.
results = pd.DataFrame(eval_rows)
results.to_csv("prompt_evaluation_results.csv", index=False)
print("Saved prompt_evaluation_results.csv", results.shape)
display(results[["prompt_version", "question_id", "latency_seconds", "input_tokens", "output_tokens",
                 "valid_json", "schema_ok"]])

### 3.5 Promotion justification

In [ ]:
m, o = metrics, metrics.drop(index=best)
best_valid = m.loc[best, "valid_json"]
decisive = best_valid > o["valid_json"].max()

def cmp(col):
    """Describe how the promoted version compares with the others on one metric."""
    parts = []
    for other in o.index:
        a, b = m.loc[best, col], m.loc[other, col]
        parts.append(("lower than " if a < b else "higher than " if a > b else "equal to ") + other)
    return " and ".join(parts)

others_valid = ", ".join(f"{k} {m.loc[k, 'valid_json']:.0%}" for k in o.index)
if decisive:
    reliability = (
        f"The decisive factor is output reliability: the EduAssist application needs strict JSON (answer, confidence, key_points) that "
        f"downstream code can parse, and {best} achieved a valid_json rate of {best_valid:.0%} against {others_valid}. "
        f"Its exact-schema compliance was {m.loc[best, 'schema_ok']:.0%}, so the answers are not only parseable but also carry the "
        f"required fields, which is the quality dimension that matters for the product. ")
elif best_valid > 0:
    reliability = (
        f"{best} reached a valid_json rate of {best_valid:.0%}, which is not higher than the others ({others_valid}), so the exact-schema "
        f"compliance ({m.loc[best, 'schema_ok']:.0%}) and then latency decided the ranking. ")
else:
    reliability = (
        f"No version produced valid JSON (all valid_json rates are 0%), so this promotion is provisional: the ranking fell back to "
        f"latency and the responses should be reviewed before production use. ")

cheapest, fastest = m["avg_cost_per_request"].idxmin(), m["avg_latency_seconds"].idxmin()
tradeoff = []
if best != cheapest:
    extra = m.loc[best, "avg_cost_per_request"] / m.loc[cheapest, "avg_cost_per_request"] - 1
    tradeoff.append(f"costs {extra:.0%} more per request than {cheapest}, the cheapest version")
if best != fastest:
    extra = m.loc[best, "avg_latency_seconds"] / m.loc[fastest, "avg_latency_seconds"] - 1
    tradeoff.append(f"is {extra:.0%} slower than {fastest}, the fastest version")
tradeoff_text = (f"The trade-off of this choice is that {best} " + " and ".join(tradeoff) + "; this is accepted because "
                 + ("reliable machine-readable output is the stated requirement" if decisive else "no other version was better on the selection rule")
                 + ". ") if tradeoff else f"{best} is also the cheapest and the fastest of the three, so there is no trade-off. "

justification = (
    f"Prompt {best} is promoted to production. " + reliability +
    f"On latency, {best} averaged {m.loc[best, 'avg_latency_seconds']:.2f} s per request, which is {cmp('avg_latency_seconds')}; "
    f"latency mainly follows output length because the model generates one token at a time. "
    f"On tokens, {best} used {m.loc[best, 'avg_input_tokens']:.1f} input tokens ({cmp('avg_input_tokens')}) and produced "
    f"{m.loc[best, 'avg_output_tokens']:.1f} output tokens ({cmp('avg_output_tokens')}). "
    f"On cost, the estimated average is {m.loc[best, 'avg_cost_per_request']:.8f} {CURRENCY} per request ({cmp('avg_cost_per_request')}), "
    f"which projects to {m.loc[best, 'avg_cost_per_request'] * REQUESTS_PER_MONTH:.4f} {CURRENCY} for {REQUESTS_PER_MONTH:,} requests a month. "
    + tradeoff_text +
    f"The selection rule was explicit and repeatable (highest valid_json, then highest schema compliance, then lowest latency), "
    f"and the alias keeps the change reversible, because production can be pointed back to an earlier version in one call."
)
n_words = len(justification.split())
print(justification)
print(f"\n[{n_words} words]")
assert n_words >= 150, "justification must be at least 150 words"

In [ ]:
print("Files written in this folder:")
for f in ["latency_tokens_cost.csv", "prompt_evaluation_results.csv", "mlflow.db"]:
    print(" ", f, "-", "found" if Path(f).exists() else "MISSING")
print("\nNext: open the MLflow UI (see README) and take the screenshots.")